# SISTEMA HIBRIDO DE MONITOREO DE GANANCIA FORESTAL - GUATEMALA
### Analisis AFOLU/REDD+ con Earth Engine

**Autor:** Juan Manuel Custodio de Leon
**Periodo:** 2020-2024

---

| # | Variable | Fuente | Peso |
|---|----------|--------|------|
| 1 | RVI anual (Sentinel-1) | S1 GRD GEE | 35 pts |
| 2 | Persistencia NDVI | S2 SR GEE | 20 pts |
| 3 | Persistencia NBR | S2 SR GEE | 15 pts |
| 4 | Persistencia EVI | S2 SR GEE | 10 pts |
| 5 | Persistencia SAVI | S2 SR GEE | 10 pts |
| 6 | Pendiente segura | SRTM 30m GEE | 10 pts |
| - | TOTAL | | 100 pts |

### Capas vectoriales
- Municipios: IGN 2016 (ArcGIS Online publico)
- Areas protegidas: SIGAP (ArcGIS Online publico)
- Cuencas 38: IGN (ArcGIS Online publico)
- Mascara Bosque/NoBosque 2022: Asset GEE propio

> **Antes de ejecutar:** edita solo la Seccion 1.

---
## SECCION 0 - Instalacion de dependencias

In [ ]:
!pip install -q earthengine-api geemap requests geopandas shapely

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 19.2 MB/s eta 0:00:00


In [ ]:
import ee
import geemap
import requests
import json
import geopandas as gpd
import pandas as pd
import numpy as np
import math

ee.Authenticate()  # Descomenta SOLO la primera vez
ee.Initialize(project='ee-manuelcustodio1010')  # <- reemplaza con tu project ID
print('Earth Engine inicializado correctamente')

Earth Engine inicializado correctamente


---
## SECCION 1 - CONFIGURACION GLOBAL
> Edita SOLO esta seccion antes de ejecutar.

In [ ]:
# ── PERIODO ──────────────────────────────────────────────────────────────────
START_YEAR = 2020
END_YEAR   = 2024

# ── ASSETS GEE (reemplaza con tus rutas reales) ───────────────────────────────
ASSET_BOSQUE_2022   = 'projects/my-project-geebook/assets/01_2022_2024_DinCobF/01_mascara_bosque_2022'    # <- edita
ASSET_NOBOSQUE_2022 = 'projects/my-project-geebook/assets/01_2022_2024_DinCobF/01_mascara_no_bosque_2022'  # <- edita

# ── CAPAS VECTORIALES (URLs publicas ArcGIS Online) ───────────────────────────
# Municipios IGN 2016
URL_MUNICIPIOS = 'https://services1.arcgis.com/1ZwElXgfK1bj5Hq0/arcgis/rest/services/Limites_Municipales_IGN_2016/FeatureServer/0'
# Areas Protegidas SIGAP
URL_AREAS_PROT = 'https://services3.arcgis.com/VFZQXvPcXdCvKIBC/arcgis/rest/services/SIGAP_junio_act/FeatureServer/0'
# Cuencas 38
URL_CUENCAS    = 'https://services1.arcgis.com/1ZwElXgfK1bj5Hq0/arcgis/rest/services/Cuencas_38/FeatureServer/0'

# ── UMBRALES DE INDICES ───────────────────────────────────────────────────────
NDVI_THRESH = 0.45
NBR_THRESH  = 0.20
EVI_THRESH  = 0.25
SAVI_THRESH = 0.25
RVI_THRESH = 0.5   # valor minimo de RVI en END_YEAR (confirma cobertura actual)
RVI_DELTA   = 0.20   # incremento minimo de RVI entre START_YEAR y END_YEAR
SLOPE_MAX   = 45.0   # pendiente maxima aceptable en grados
ELEV_MIN    = 0      # elevacion minima (m)
ELEV_MAX    = 3500   # elevacion maxima (m)

# ── PESOS DE PONDERACION (deben sumar 100) ────────────────────────────────────
PESO_RVI       = 35
PESO_NDVI      = 25
PESO_NBR       = 0
PESO_EVI       = 15
PESO_SAVI      = 15
PESO_PENDIENTE = 10
# TOTAL        = 100

# Umbral minimo de score para clasificar como ganancia confirmada
SCORE_THRESH = 65

# ── EXPORTACION ───────────────────────────────────────────────────────────────
EXPORT_SCALE   = 10
EXPORT_MAX_PIX = 1e13
EXPORT_FOLDER  = 'GEE_Ganancias_Forestales_2020_2024'

# Regiones REDD+
ASSET_REDD_REGIONS = 'users/inabsigguatemala/InformacionBase/Propuesta_subnac_un2021_REDD'

# Regiones a procesar (comenta/descomenta segun necesites)
REGIONES_ACTIVAS = [
    'TBN',
    'SARSTUN_MOTAGUA',
    'CENTRO_ORIENTE',
    'OCCIDENTE',
    'COSTA_SUR'
]

# Mapeo nombre Region_sub -> clave interna
REGION_NAME_MAP = {
    'Tierras Bajas del Norte': 'TBN',
    'Sarstún Motagua':         'SARSTUN_MOTAGUA',
    'Centro Oriente':          'CENTRO_ORIENTE',
    'Occidente':               'OCCIDENTE',
    'Costa Sur':               'COSTA_SUR'
}

print('Configuracion cargada')
print('Pesos -> RVI:' + str(PESO_RVI) + ' NDVI:' + str(PESO_NDVI) +
      ' NBR:' + str(PESO_NBR) + ' EVI:' + str(PESO_EVI) +
      ' SAVI:' + str(PESO_SAVI) + ' Pendiente:' + str(PESO_PENDIENTE) +
      ' | TOTAL:' + str(PESO_RVI+PESO_NDVI+PESO_NBR+PESO_EVI+PESO_SAVI+PESO_PENDIENTE))

Configuracion cargada
Pesos -> RVI:35 NDVI:25 NBR:0 EVI:15 SAVI:15 Pendiente:10 | TOTAL:100


---
## SECCION 2 - CARGA DE CAPAS VECTORIALES

Las tres capas son servicios publicos de ArcGIS Online — no requieren token.
Se descargan via REST API con paginacion automatica y se convierten a `ee.FeatureCollection`.

In [ ]:
def clean_geojson_for_ee(geojson):
    cleaned = []
    for i, feat in enumerate(geojson['features']):
        props = {}
        for k, v in feat.get('properties', {}).items():
            if v is None:
                props[str(k)] = ''
            elif isinstance(v, (int, float, str, bool)):
                props[str(k)] = v
            else:
                props[str(k)] = str(v)
        props['system:index'] = str(i)
        cleaned.append({
            'type'      : 'Feature',
            'geometry'  : feat.get('geometry'),
            'properties': props
        })
    return {'type': 'FeatureCollection', 'features': cleaned}


def fetch_arcgis_public(base_url, where='1=1', out_fields='*', page_size=1000):
    all_features = []
    offset = 0
    while True:
        params = {
            'where': where, 'outFields': out_fields,
            'outSR': '4326', 'f': 'geojson',
            'resultRecordCount': page_size, 'resultOffset': offset,
            'returnGeometry': 'true'
        }
        resp = requests.get(base_url + '/query', params=params, timeout=90)
        resp.raise_for_status()
        features = resp.json().get('features', [])
        if not features:
            break
        all_features.extend(features)
        offset += len(features)
        if len(features) < page_size:
            break

    if not all_features:
        raise ValueError('Sin features en: ' + base_url)

    geojson_clean = clean_geojson_for_ee({'type': 'FeatureCollection', 'features': all_features})
    gdf = gpd.GeoDataFrame.from_features(geojson_clean['features'], crs='EPSG:4326')
    fc  = ee.FeatureCollection(geojson_clean)
    return gdf, fc

In [ ]:
print('Cargando Municipios IGN 2016...')
gdf_municipios, fc_municipios = fetch_arcgis_public(URL_MUNICIPIOS)
print('  OK: ' + str(len(gdf_municipios)) + ' municipios')
print('  Columnas: ' + str(list(gdf_municipios.columns)))

print('Cargando Areas Protegidas SIGAP...')
gdf_sigap, fc_sigap = fetch_arcgis_public(URL_AREAS_PROT)
print('  OK: ' + str(len(gdf_sigap)) + ' areas protegidas')
print('  Columnas: ' + str(list(gdf_sigap.columns)))

print('Cargando Cuencas 38...')
gdf_cuencas, fc_cuencas = fetch_arcgis_public(URL_CUENCAS)
print('  OK: ' + str(len(gdf_cuencas)) + ' cuencas')
print('  Columnas: ' + str(list(gdf_cuencas.columns)))

print()
print('Todas las capas vectoriales cargadas')
print('IMPORTANTE: revisa las columnas impresas arriba y ajusta')
print('CAMPO_MUNICIPIO en la Seccion 12 si es necesario.')

Cargando Municipios IGN 2016...
  OK: 340 municipios
  Columnas: ['geometry', 'FID', 'cod_dept_1', 'cod_muni_1', 'nombre_1', 'depto_1', 'area_km__', 'perimetr_1', 'ha', 'SHAPE__Len', 'SHAPE__Are', 'idoneidad', 'Shape__Area', 'Shape__Length', 'system:index']
Cargando Areas Protegidas SIGAP...
  OK: 406 areas protegidas
  Columnas: ['geometry', 'FID', 'codigo_g_1', 'codigo_e_2', 'NOMBRE_G_1', 'Categor_13', 'NOMBRE_e_1', 'Categor_14', 'Tipo_Categ', 'Shape_Leng', 'Shape_Area', 'Shape__Area', 'Shape__Length', 'system:index']
Cargando Cuencas 38...
  OK: 38 cuencas
  Columnas: ['geometry', 'FID', 'codigo', 'label', 'vertiente', 'km2', 'ha', 'name', 'padre', 'cod', 'SHAPE__Len', 'SHAPE__Are', 'Shape__Area', 'Shape__Length', 'system:index']

Todas las capas vectoriales cargadas
IMPORTANTE: revisa las columnas impresas arriba y ajusta
CAMPO_MUNICIPIO en la Seccion 12 si es necesario.


---
## SECCION 3 - COLECCIONES SATELITALES Y MASCARAS BASE

In [ ]:
# Mascara bosque / no bosque 2020
bosque_2022   = ee.Image(ASSET_BOSQUE_2022).selfMask()
nobosque_2022 = ee.Image(ASSET_NOBOSQUE_2022).selfMask()
print('Mascaras 2022 cargadas')

# DEM SRTM
dem       = ee.Image('USGS/SRTMGL1_003')
slope_deg = ee.Terrain.slope(dem)
# Pendiente en porcentaje: tan(grados * pi/180) * 100
slope_pct = slope_deg.multiply(math.pi / 180).tan().multiply(100).rename('slope_pct')
elevation = dem.rename('elevation')
print('DEM, pendiente y elevacion listos')

# Sentinel-2 SR Harmonized
def mask_s2_scl(image):
    """Mascara de nubes y sombras usando SCL.
    Excluye: 3=sombra, 8=nube media, 9=nube alta, 10=cirros, 11=nieve
    """
    scl = image.select('SCL')
    mask = (scl.neq(3).And(scl.neq(8)).And(scl.neq(9))
               .And(scl.neq(10)).And(scl.neq(11)))
    return image.updateMask(mask).divide(10000)  # escalar a 0-1

s2_base = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
             .filterDate(str(START_YEAR) + '-01-01', str(END_YEAR) + '-12-31')
             .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 70))
             .map(mask_s2_scl))

# Sentinel-1 GRD Descendente
s1_base = (ee.ImageCollection('COPERNICUS/S1_GRD')
             .filterDate(str(START_YEAR) + '-01-01', str(END_YEAR) + '-12-31')
             .filter(ee.Filter.eq('instrumentMode', 'IW'))
             .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))
             .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
             .filter(ee.Filter.eq('orbitProperties_pass', 'DESCENDING')))

print('Colecciones S1 y S2 inicializadas')

Mascaras 2022 cargadas
DEM, pendiente y elevacion listos
Colecciones S1 y S2 inicializadas


---
## SECCION 4 - INDICES ESPECTRALES Y PERSISTENCIA TEMPORAL

**Indices calculados sobre Sentinel-2 SR (bandas escaladas 0-1):**
- **NDVI** = (NIR - Rojo) / (NIR + Rojo)
- **NBR** = (NIR - SWIR2) / (NIR + SWIR2)
- **EVI** = 2.5 * (NIR - Rojo) / (NIR + 6*Rojo - 7.5*Azul + 1)
- **SAVI** = ((NIR - Rojo) / (NIR + Rojo + 0.5)) * 1.5

In [ ]:
def annual_s2_median(year, aoi):
    """
    Mediana anual S2. Si la coleccion queda vacia con el filtro estricto,
    relaja progresivamente hasta encontrar imagenes.
    """
    filtros = [
        (str(year) + '-01-01', str(year) + '-12-31', 10),  # año completo, 70% nubes
        (str(year) + '-01-01', str(year) + '-12-31', 90),  # año completo, 90% nubes
        (str(year) + '-01-01', str(year) + '-12-31', 100), # año completo, sin filtro nubes
    ]

    for fecha_ini, fecha_fin, max_nubes in filtros:
        col = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
                 .filterDate(fecha_ini, fecha_fin)
                 .filterBounds(aoi)
                 .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', max_nubes))
                 .map(mask_s2_scl))

        # Verifica en Python si hay imagenes antes de continuar
        n = col.size().getInfo()
        if n > 0:
            return col.median().clip(aoi)

    # Si absolutamente no hay nada, devuelve imagen vacia con bandas correctas
    # para que compute_indices no truene
    return ee.Image.constant(0).rename('B2') \
             .addBands(ee.Image.constant(0).rename('B4')) \
             .addBands(ee.Image.constant(0).rename('B8')) \
             .addBands(ee.Image.constant(0).rename('B12')) \
             .selfMask()  # mascara todo para que no aparezca en el mapa

def compute_indices(image):
    """Calcula NDVI, NBR, EVI y SAVI sobre imagen S2 escalada (0-1)."""
    nir  = image.select('B8')
    red  = image.select('B4')
    blue = image.select('B2')

    ndvi = image.normalizedDifference(['B8', 'B4']).rename('NDVI')
    nbr  = image.normalizedDifference(['B8', 'B12']).rename('NBR')
    evi  = image.expression(
        '2.5 * ((NIR - RED) / (NIR + 6.0 * RED - 7.5 * BLUE + 1.0))',
        {'NIR': nir, 'RED': red, 'BLUE': blue}
    ).rename('EVI')
    savi = image.expression(
        '((NIR - RED) / (NIR + RED + 0.5)) * 1.5',
        {'NIR': nir, 'RED': red}
    ).rename('SAVI')

    return ee.Image.cat([ndvi, nbr, evi, savi])


def annual_indices(year, aoi):
    """Devuelve NDVI, NBR, EVI, SAVI. Imprime advertencia si no hay datos S2."""
    s2 = annual_s2_median(year, aoi)
    indices = compute_indices(s2)
    return indices


def calculate_persistence(aoi, n_years, index_name, threshold):
    """
    Evalua cuantos de los ultimos n_years el indice supero el umbral.
    Devuelve:
      binary : ee.Image 0/1 - True si supero en TODOS los anos
      count  : ee.Image - numero de anos que supero el umbral
    """
    years = list(range(END_YEAR - n_years + 1, END_YEAR + 1))
    imgs = []
    for yr in years:
        idx = annual_indices(yr, aoi).select(index_name)
        imgs.append(idx.gt(threshold).unmask(0))
    count  = ee.ImageCollection(imgs).sum().rename(index_name + '_count')
    binary = count.gte(n_years).rename(index_name + '_persistent')
    return binary, count


print('Indices espectrales y persistencia definidos')

Indices espectrales y persistencia definidos


---
## SECCION 5 - RVI SENTINEL-1 (Radar Vegetation Index)

**Formula:** `RVI = 4 * VH_lin / (VV_lin + VH_lin)` (Kim & van Zyl 2009)

Los valores S1 GRD estan en dB y se convierten a potencia lineal antes del calculo.
Valores altos (cercanos a 1) indican vegetacion densa con estructura vertical.

In [ ]:
def db_to_linear(db_image):
    """Convierte backscatter de dB a potencia lineal."""
    return db_image.expression('10.0 ** (db / 10.0)', {'db': db_image})


def annual_rvi(year, aoi):
    """RVI mediano anual para un ano y area dados."""
    def add_rvi(image):
        vh = db_to_linear(image.select('VH'))
        vv = db_to_linear(image.select('VV'))
        rvi = vh.multiply(4.0).divide(vv.add(vh)).rename('RVI')
        return image.addBands(rvi)

    return (s1_base
            .filterDate(str(year) + '-06-01', str(year) + '-12-31')
            .filterBounds(aoi)
            .map(add_rvi)
            .select('RVI')
            .median()
            .clip(aoi))


def calculate_rvi_increase(aoi):
    """
    Evalua incremento RVI entre START_YEAR y END_YEAR.
    La mascara es True solo si SE CUMPLEN AMBAS condiciones:
      1. delta RVI > RVI_DELTA  (hubo crecimiento estructural)
      2. RVI final > RVI_THRESH (la cobertura actual es significativa)
    """
    rvi_ini = annual_rvi(START_YEAR, aoi).rename('RVI_inicio')
    rvi_fin = annual_rvi(END_YEAR,   aoi).rename('RVI_fin')
    delta   = rvi_fin.subtract(rvi_ini).rename('delta_RVI')

    mask_delta  = delta.gt(RVI_DELTA)
    mask_thresh = rvi_fin.gt(RVI_THRESH)

    mask = mask_delta.And(mask_thresh).rename('RVI_mask')
    return mask, delta, rvi_ini, rvi_fin


print('Funciones RVI definidas')

Funciones RVI definidas


---
## SECCION 6 - FILTROS TOPOGRAFICOS (Pendiente y Elevacion)

In [ ]:
def build_topographic_masks(aoi):
    """
    Genera mascaras binarias de pendiente y elevacion.
    Returns:
      slope_mask : 1 donde pendiente <= SLOPE_MAX
      elev_mask  : 1 donde elevacion en [ELEV_MIN, ELEV_MAX]
      slope_img  : pendiente en % recortada al AOI
      elev_img   : elevacion en m recortada al AOI
    """
    sl = slope_pct.clip(aoi)
    el = elevation.clip(aoi)
    thresh_pct = math.tan(math.radians(SLOPE_MAX)) * 100
    slope_mask = sl.lte(thresh_pct).rename('slope_ok')
    elev_mask  = el.gte(ELEV_MIN).And(el.lte(ELEV_MAX)).rename('elev_ok')
    return slope_mask, elev_mask, sl, el


print('Filtros topograficos listos')
print('  Pendiente maxima : ' + str(SLOPE_MAX) + ' grados (~' +
      str(round(math.tan(math.radians(SLOPE_MAX)) * 100, 1)) + '%)')
print('  Elevacion valida : ' + str(ELEV_MIN) + ' - ' + str(ELEV_MAX) + ' m')

Filtros topograficos listos
  Pendiente maxima : 45.0 grados (~100.0%)
  Elevacion valida : 0 - 3500 m


---
## SECCION 7 - PONDERACION Y SCORE FINAL

| Variable | Peso |
|----------|------|
| Incremento RVI (SAR estructural) | 35 pts |
| Persistencia NDVI | 20 pts |
| Persistencia NBR | 15 pts |
| Persistencia EVI | 10 pts |
| Persistencia SAVI | 10 pts |
| Pendiente segura | 10 pts |
| **TOTAL** | **100 pts** |

**Categorias de ganancia:**
- 4 = ALTA (80-100 pts)
- 3 = MEDIA (65-79 pts)
- 2 = BAJA (50-64 pts)
- 1 = NO CONFIRMADA (< 50 pts)

In [ ]:
def build_weighted_score(p_ndvi, p_nbr, p_evi, p_savi, rvi_mask, slope_mask):
    """
    Combina todas las variables binarias (0/1) con sus pesos para
    generar un score continuo de 0 a 100.
    """
    score = (
        rvi_mask  .multiply(PESO_RVI)
        .add(p_ndvi    .multiply(PESO_NDVI))
        .add(p_nbr     .multiply(PESO_NBR))
        .add(p_evi     .multiply(PESO_EVI))
        .add(p_savi    .multiply(PESO_SAVI))
        .add(slope_mask.multiply(PESO_PENDIENTE))
    ).rename('score_total')
    return score


def categorize_score(score_img):
    """Clasifica el score en 4 categorias de ganancia."""
    cat = (ee.Image(1)
           .where(score_img.gte(50).And(score_img.lt(65)), 2)
           .where(score_img.gte(65).And(score_img.lt(80)), 3)
           .where(score_img.gte(80),                       4)
           ).rename('categoria')
    return cat


print('Funciones de ponderacion y categorizacion definidas')

Funciones de ponderacion y categorizacion definidas


---
## SECCION 8 - PROCESAMIENTO PRINCIPAL

> Los calculos en GEE son diferidos (lazy). Esta celda construye el grafo de
> computo; la exportacion o la visualizacion son las que lo materializan.

In [ ]:
# Carga regiones REDD+
redd_regions = ee.FeatureCollection(ASSET_REDD_REGIONS)

# Filtra solo las regiones activas
region_names_sub = [k for k, v in REGION_NAME_MAP.items() if v in REGIONES_ACTIVAS]
redd_activas = redd_regions.filter(ee.Filter.inList('Region_sub', region_names_sub))

N_PERS = 3  # anos de persistencia (ajustable)

resultados = {}  # guarda outputs por region para exportacion

region_list = redd_activas.toList(redd_activas.size()).getInfo()

for feat in region_list:
    region_sub  = feat['properties']['Region_sub']
    region_key  = REGION_NAME_MAP.get(region_sub)

    if not region_key:
        print('Sin mapeo para: ' + region_sub)
        continue

    print('Procesando: ' + region_sub + ' (' + region_key + ')')
    aoi = ee.Feature(feat).geometry()

    # Persistencia espectral
    p_ndvi, _ = calculate_persistence(aoi, N_PERS, 'NDVI', NDVI_THRESH)
    p_nbr,  _ = calculate_persistence(aoi, N_PERS, 'NBR',  NBR_THRESH)
    p_evi,  _ = calculate_persistence(aoi, N_PERS, 'EVI',  EVI_THRESH)
    p_savi, _ = calculate_persistence(aoi, N_PERS, 'SAVI', SAVI_THRESH)

    # RVI
    rvi_mask, delta_rvi, rvi_ini, rvi_fin = calculate_rvi_increase(aoi)

    # Topografia
    slope_mask, elev_mask, slope_img, elev_img = build_topographic_masks(aoi)

    # Candidatos (no-bosque 2022)
    candidates = nobosque_2022.clip(aoi)

    # Score
    score = build_weighted_score(
        p_ndvi, p_nbr, p_evi, p_savi, rvi_mask, slope_mask
    ).updateMask(candidates)

    # Categorizacion y ganancia
    categoria      = categorize_score(score)
    ganancia_final = score.gte(SCORE_THRESH).selfMask().rename('ganancia_confirmada')

    resultados[region_key] = {
        'aoi'           : aoi,
        'score'         : score,
        'categoria'     : categoria,
        'ganancia_final': ganancia_final,
        'delta_rvi'     : delta_rvi,
        'slope_img'     : slope_img,
        'elev_img'      : elev_img,
    }
    print('  OK - ' + region_key)

print()
print('Regiones procesadas: ' + str(list(resultados.keys())))

Procesando: Tierras Bajas del Norte (TBN)
  OK - TBN
Procesando: Occidente (OCCIDENTE)
  OK - OCCIDENTE
Procesando: Centro Oriente (CENTRO_ORIENTE)
  OK - CENTRO_ORIENTE
Procesando: Sarstún Motagua (SARSTUN_MOTAGUA)
  OK - SARSTUN_MOTAGUA
Procesando: Costa Sur (COSTA_SUR)
  OK - COSTA_SUR

Regiones procesadas: ['TBN', 'OCCIDENTE', 'CENTRO_ORIENTE', 'SARSTUN_MOTAGUA', 'COSTA_SUR']


---
## SECCION 9 - VISUALIZACION EN MAPA

> Esta celda puede tardar unos segundos en renderizar cada capa.

In [ ]:
# Region a visualizar en el mapa (cambia segun necesites)
REGION_VIZ = 'COSTA_SUR'  # <- cambia a la region que quieras revisar

r   = resultados[REGION_VIZ]
aoi_viz = r['aoi']

Map = geemap.Map()
Map.centerObject(aoi_viz, 9)

Map.addLayer(bosque_2022,           {'palette': ['006400']}, 'Bosque 2022', False)
Map.addLayer(nobosque_2022.clip(aoi_viz), {'palette': ['FFA500']}, 'NoBosque 2022', False)

idx_fin = annual_indices(END_YEAR, aoi_viz)
Map.addLayer(idx_fin.select('NDVI'), {'min': 0.0, 'max': 1.0, 'palette': ['white','green']},      'NDVI ' + str(END_YEAR), False)
Map.addLayer(idx_fin.select('NBR'),  {'min': -0.5, 'max': 1.0, 'palette': ['red','white','green']},'NBR '  + str(END_YEAR), False)
Map.addLayer(idx_fin.select('EVI'),  {'min': 0.0, 'max': 1.0, 'palette': ['white','darkgreen']},  'EVI '  + str(END_YEAR), False)
Map.addLayer(idx_fin.select('SAVI'), {'min': 0.0, 'max': 1.0, 'palette': ['white','olive']},      'SAVI ' + str(END_YEAR), False)

Map.addLayer(r['delta_rvi'], {'min': -0.3, 'max': 0.5, 'palette': ['red','white','blue']}, 'Delta RVI', False)
Map.addLayer(r['slope_img'], {'min': 0, 'max': 100,    'palette': ['green','yellow','red']}, 'Pendiente (%)', False)
Map.addLayer(r['elev_img'],  {'min': 0, 'max': 4000,   'palette': ['green','yellow','brown','white']}, 'Elevacion (m)', False)

Map.addLayer(r['score'],
             {'min': 0, 'max': 100, 'palette': ['#E60000','#FF6B00','#FFD600','#00E676','#0055FF']},
             'Score - ' + REGION_VIZ, False)
Map.addLayer(r['categoria'],
             {'min': 1, 'max': 4, 'palette': ['#d9d9d9','#fdae61','#1a9641','#004529']},
             'Categorias - ' + REGION_VIZ)
Map.addLayer(r['ganancia_final'],
             {'palette': ['4927F5']},
             'Ganancia - ' + REGION_VIZ)



Map.add_legend(
    title='Categoria de Ganancia',
    legend_dict={
        '4 - Alta (80-100)':    '#004529',
        '3 - Media (65-79)':    '#1a9641',
        '2 - Baja (50-64)':     '#fdae61',
        '1 - No confirmada':    '#d9d9d9',
    }
)
Map

Map(center=[14.31109804087046, -91.13913496111473], controls=(WidgetControl(options=['position', 'transparent_…

---
## SECCION 10 - EXPORTACION A GOOGLE DRIVE Y ASSETS GEE

In [ ]:
def export_region(image, region_key, desc_prefix, file_prefix, dtype='float'):
    img  = image.toFloat() if dtype == 'float' else image.toByte()
    task = ee.batch.Export.image.toDrive(
        image          = img,
        description    = desc_prefix + '_' + region_key,
        folder         = EXPORT_FOLDER,
        fileNamePrefix = file_prefix + '_' + region_key,
        region         = resultados[region_key]['aoi'],
        scale          = EXPORT_SCALE,
        maxPixels      = EXPORT_MAX_PIX,
        crs            = 'EPSG:4326'
    )
    task.start()
    return task


print('Iniciando exportaciones por region...')
print()

for region_key, r in resultados.items():
    print('Exportando: ' + region_key)

    export_region(r['score'],         region_key, 'Score',     '01_score',     'float')
    export_region(r['categoria'],     region_key, 'Categorias','02_categorias','byte')
    export_region(r['ganancia_final'],region_key, 'Ganancia',  '03_ganancia',  'byte')
    export_region(r['delta_rvi'],     region_key, 'DeltaRVI',  '04_delta_rvi', 'float')

    # Asset GEE por region
  #  task_asset = ee.batch.Export.image.toAsset(
   #     image       = r['ganancia_final'].gt(0).toByte(),
    #    description = 'ASSET_Ganancia_' + region_key,
     #   assetId     = 'projects/ee-manuelcustodio1010/assets/Ganancias_' + region_key + '_' + str(START_YEAR) + '_' + str(END_YEAR),
      #  region      = r['aoi'],
       # scale       = EXPORT_SCALE,
       # maxPixels   = EXPORT_MAX_PIX
  #  )
   # task_asset.start()

    print('  OK - 4 archivos Drive + 1 Asset GEE')

print()
print('Total regiones exportadas: ' + str(len(resultados)))
print('Revisa el estado en: https://code.earthengine.google.com/ > Tasks')

Iniciando exportaciones por region...

Exportando: TBN
  OK - 4 archivos Drive + 1 Asset GEE
Exportando: OCCIDENTE
  OK - 4 archivos Drive + 1 Asset GEE
Exportando: CENTRO_ORIENTE
  OK - 4 archivos Drive + 1 Asset GEE
Exportando: SARSTUN_MOTAGUA
  OK - 4 archivos Drive + 1 Asset GEE
Exportando: COSTA_SUR
  OK - 4 archivos Drive + 1 Asset GEE

Total regiones exportadas: 5
Revisa el estado en: https://code.earthengine.google.com/ > Tasks


---
## SECCION 11 - ESTADISTICAS DE AREA POR MUNICIPIO (OPCIONAL)

In [ ]:
# Ajusta al nombre exacto de la columna de nombre en tu capa de municipios.
# La celda anterior imprimio las columnas disponibles — revisalas.
CAMPO_MUNICIPIO = 'nombre_mun'  # <- ajusta si es necesario

print('Calculando area de ganancia por municipio...')

pixel_area = ee.Image.pixelArea().divide(10000)  # hectareas
gain_area  = ganancia_final.multiply(pixel_area)

stats = gain_area.reduceRegions(
    collection=fc_municipios,
    reducer=ee.Reducer.sum().setOutputs(['area_ha']),
    scale=EXPORT_SCALE
)

info = stats.select([CAMPO_MUNICIPIO, 'area_ha']).getInfo()

rows = []
for f in info['features']:
    rows.append({
        'municipio': f['properties'].get(CAMPO_MUNICIPIO, 'N/A'),
        'area_ha'  : round(f['properties'].get('area_ha', 0), 2)
    })

df = pd.DataFrame(rows).sort_values('area_ha', ascending=False)
df = df[df['area_ha'] > 0].reset_index(drop=True)

print('Municipios con ganancia confirmada: ' + str(len(df)))
print('Area total: ' + str(round(df['area_ha'].sum(), 1)) + ' ha')
print()
print(df.head(20).to_string(index=False))

In [ ]:
csv_name = 'ganancia_municipios_' + str(START_YEAR) + '_' + str(END_YEAR) + '.csv'
df.to_csv(csv_name, index=False, encoding='utf-8-sig')
print('Guardado: ' + csv_name)

from google.colab import files
files.download(csv_name)

---
## SECCION 12 - SERIE TEMPORAL RVI POR POLIGONO (OPCIONAL)

Util para validar el comportamiento de un area especifica antes de confirmar ganancia.

In [ ]:
import matplotlib.pyplot as plt


def plot_rvi_series(geometry, label='Poligono'):
    """
    Grafica la serie anual de RVI para una geometria.
    Args:
        geometry : ee.Geometry o .geometry() de un ee.Feature
        label    : nombre para el grafico y archivo PNG
    """
    years  = list(range(START_YEAR, END_YEAR + 1))
    values = []
    for yr in years:
        val = (annual_rvi(yr, geometry)
               .reduceRegion(
                   reducer=ee.Reducer.mean(),
                   geometry=geometry,
                   scale=10,
                   maxPixels=1e9)
               .get('RVI').getInfo())
        values.append(val if val is not None else float('nan'))

    plt.figure(figsize=(10, 4))
    plt.plot(years, values, marker='o', color='steelblue', linewidth=2, markersize=7)
    plt.fill_between(years, values, alpha=0.15, color='steelblue')
    plt.axhline(y=RVI_DELTA, color='red', linestyle='--', alpha=0.7,
                label='Umbral delta RVI (' + str(RVI_DELTA) + ')')
    plt.title('Serie temporal RVI - ' + label, fontsize=13)
    plt.xlabel('Ano'); plt.ylabel('RVI promedio')
    plt.ylim(0, 1); plt.grid(True, alpha=0.3); plt.legend()
    plt.tight_layout()
    plt.savefig('rvi_serie_' + label.replace(' ', '_') + '.png', dpi=150)
    plt.show()
    if not all(math.isnan(v) for v in values):
        print('Delta RVI (' + str(START_YEAR) + '->' + str(END_YEAR) + '): ' +
              str(round(values[-1] - values[0], 3)))


# Ejemplo de uso: descomenta las lineas siguientes para graficar
# el primer municipio de la coleccion
# primer_mun = ee.Feature(fc_municipios.first())
# plot_rvi_series(primer_mun.geometry(), label='Primer_Municipio')

print('Funcion plot_rvi_series lista')
print('Descomenta las ultimas 2 lineas para graficar un area especifica')